In [4]:
"""
Mart-слой: Random Forest регрессия длительности матча по признакам на 15-й минуте.

Источник: stage.dota_duration_15min_match_features
Результат: mart.dota_duration_15min_match_duration_predictions

Колонки предсказаний:
  baseline_pred — средняя длительность по train (baseline)
  tree_pred     — прогноз Random Forest
"""

import time
import requests

from lakehouse_spark import SparkSessionContext
from lakehouse_publish import publish_table

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F
from pyspark import StorageLevel
import pandas as pd

# ===================================================================
# Признаки и таргет
# ===================================================================
feature_cols = [
    "game_mode",
    "first_blood_time",
    "first_tower_time", "has_tower_15m", "total_towers_15m",
    "kills_15m", "kills_radiant_15m", "kills_dire_15m", "kills_diff_15m",
    "gold_radiant_15m", "gold_dire_15m", "gold_diff_15m", "abs_gold_diff_15m",
]
label_col = "duration"

config = {
    "spark.sql.parquet.enableVectorizedReader": "false",
    "spark.sql.iceberg.vectorization.enabled": "false",
}


# ===================================================================
# Retry-обёртка над publish_table (portal-api бывает медленным)
# ===================================================================
def publish_with_retry(layer, table_name, dataframe, description, dependencies,
                       max_attempts=5, base_delay=5):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"[publish] attempt {attempt}/{max_attempts}: {layer}.{table_name}")
            return publish_table(
                layer, table_name, dataframe,
                description=description, dependencies=dependencies,
            )
        except (requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectionError) as e:
            last_exc = e
            if attempt == max_attempts:
                break
            delay = base_delay * (2 ** (attempt - 1))
            print(f"[publish] attempt {attempt} failed: {e!r}. Retry in {delay}s...")
            time.sleep(delay)

    print(
        f"[publish] ALL {max_attempts} attempts failed for {layer}.{table_name}. "
        f"Таблица УЖЕ записана в lakehouse, но НЕ зарегистрирована в portal. "
        f"Дозвонить вручную: publish_table('{layer}', '{table_name}', "
        f"spark.table('lakehouse.{layer}.{table_name}'), "
        f"description='...', dependencies={dependencies}). "
        f"Last error: {last_exc!r}"
    )
    return None


with SparkSessionContext("dota-duration-15min-mart", config=config) as ctx:
    spark = ctx

    pipeline = Pipeline(stages=[
        VectorAssembler(
            inputCols=feature_cols,
            outputCol="features",
            handleInvalid="skip",
        ),
        RandomForestRegressor(
            featuresCol="features",
            labelCol=label_col,
            predictionCol="prediction",
            numTrees=50,
            maxDepth=5,
            seed=42,
        ),
    ])

    with ctx.lineage(
        "dota-duration-15min-rf-regression",
        inputs=["stage.dota_duration_15min_match_features"],
        output="mart.dota_duration_15min_match_duration_predictions",
        description=(
            "Random Forest регрессия: предсказание длительности матча "
            "по признакам на 15-й минуте (kills, gold, towers, first blood, game_mode). "
            "Содержит baseline_pred (среднее по train) и tree_pred (прогноз RF)."
        ),
        dataset_descriptions={
            "mart.dota_duration_15min_match_duration_predictions":
                "Предсказания длительности матча: baseline_pred (среднее по train) "
                "и tree_pred (Random Forest) на одних и тех же тестовых матчах + "
                "факт, абсолютные ошибки и разметка по срезу длительности."
        },
        config={"numTrees": 50, "maxDepth": 5, "seed": 42, "split": "0.8/0.2"},
    ):
        # ---------- Подготовка ----------
        source = (
            spark.table("lakehouse.stage.dota_duration_15min_match_features")
                .select("match_id", label_col, *feature_cols)
                .dropna(subset=[label_col] + feature_cols)
                .repartition(4)
                .persist(StorageLevel.MEMORY_AND_DISK)
        )
        print("Training rows (before split):", source.count())

        train, test = source.randomSplit([0.8, 0.2], seed=42)
        train = train.persist(StorageLevel.MEMORY_AND_DISK)
        test  = test.persist(StorageLevel.MEMORY_AND_DISK)
        print(f"Train: {train.count()}, Test: {test.count()}")

        # ---------- Baseline (среднее по train) ----------
        avg_duration = train.select(F.avg(label_col)).collect()[0][0]
        print(f"Baseline avg duration: {avg_duration:.2f} сек ({avg_duration/60:.2f} мин.)")

        # ---------- Обучение ----------
        model = pipeline.fit(train)

        # ---------- Предсказания: baseline_pred + tree_pred ----------
        result = (
            model.transform(test)
                .withColumn("baseline_pred", F.lit(float(avg_duration)))
                .withColumn("tree_pred",     F.col("prediction").cast("double"))
                .withColumn("abs_error_sec",      F.abs(F.col(label_col) - F.col("tree_pred")))
                .withColumn("abs_error_baseline", F.abs(F.col(label_col) - F.col("baseline_pred")))
                .withColumn(
                    "duration_bucket",
                    F.when(F.col(label_col) < 1800, "Short (<30m)")
                     .when(F.col(label_col) < 2700, "Medium (30-45m)")
                     .otherwise("Long (>45m)")
                )
                .withColumn("model_version", F.lit("dota_rf_duration_v2"))
                .select(
                    "match_id",
                    F.col(label_col).alias("actual_duration_sec"),
                    "baseline_pred",
                    "tree_pred",
                    "abs_error_sec",
                    "abs_error_baseline",
                    "duration_bucket",
                    "model_version",
                    *feature_cols,
                )
        )

        # ---------- Запись в mart ----------
        result.writeTo("lakehouse.mart.dota_duration_15min_match_duration_predictions") \
              .using("iceberg").createOrReplace()
        print("✅ mart.dota_duration_15min_match_duration_predictions записана")

        # ---------- publish с retry ----------
        publish_with_retry(
            "mart",
            "dota_duration_15min_match_duration_predictions",
            result,
            description="Предсказания длительности матча на 15-й минуте (baseline + RF)",
            dependencies=["stage/dota_duration_15min_match_features"],
        )

    # ===================================================================
    # Метрики: tree vs baseline на одних и тех же тестовых строках
    # ===================================================================
    def rmse_mae(pred_col):
        r = RegressionEvaluator(labelCol="actual_duration_sec",
                                predictionCol=pred_col, metricName="rmse").evaluate(result)
        m = RegressionEvaluator(labelCol="actual_duration_sec",
                                predictionCol=pred_col, metricName="mae").evaluate(result)
        return r, m

    base_rmse, base_mae = rmse_mae("baseline_pred")
    tree_rmse, tree_mae = rmse_mae("tree_pred")
    tree_r2 = RegressionEvaluator(labelCol="actual_duration_sec",
                                  predictionCol="tree_pred",
                                  metricName="r2").evaluate(result)

    print("\n=== Метрики на тесте ===")
    print(f"Baseline  RMSE: {base_rmse:.2f} сек ({base_rmse/60:.2f} мин.)  MAE: {base_mae:.2f} сек ({base_mae/60:.2f} мин.)")
    print(f"RandomForest RMSE: {tree_rmse:.2f} сек ({tree_rmse/60:.2f} мин.)  MAE: {tree_mae:.2f} сек ({tree_mae/60:.2f} мин.)")
    print(f"RandomForest R²:   {tree_r2:.4f}")
    print(f"Δ RMSE (tree vs baseline): {base_rmse - tree_rmse:+.2f} сек "
          f"({'лучше' if tree_rmse < base_rmse else 'хуже'} baseline)")

    # ===================================================================
    # MAE по срезам длительности — tree vs baseline
    # ===================================================================
    print("\n=== MAE по срезам длительности (сек / мин) ===")
    result.groupBy("duration_bucket").agg(
        F.count("*").alias("matches"),
        F.round(F.avg("abs_error_sec") / 60, 2).alias("tree_MAE_min"),
        F.round(F.avg("abs_error_baseline") / 60, 2).alias("baseline_MAE_min"),
        F.round(F.avg("actual_duration_sec") / 60, 2).alias("avg_actual_min"),
    ).orderBy("duration_bucket").show(truncate=False)

    # ===================================================================
    # Важность фичей
    # ===================================================================
    rf_stage = model.stages[-1]
    importance = pd.DataFrame({
        "feature":    feature_cols,
        "importance": rf_stage.featureImportances.toArray(),
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    print("\n=== Топ фичей по важности ===")
    print(importance.head(15).to_string(index=False))

    # ===================================================================
    # Очистка кэша
    # ===================================================================
    source.unpersist(blocking=True)
    train.unpersist(blocking=True)
    test.unpersist(blocking=True)

Training rows (before split): 13048
Train: 10532, Test: 2516
Baseline avg duration: 2182.23 сек (36.37 мин.)


✅ mart.dota_duration_15min_match_duration_predictions записана
[publish] attempt 1/5: mart.dota_duration_15min_match_duration_predictions

=== Метрики на тесте ===
Baseline  RMSE: 666.19 сек (11.10 мин.)  MAE: 537.15 сек (8.95 мин.)
RandomForest RMSE: 471.58 сек (7.86 мин.)  MAE: 360.61 сек (6.01 мин.)
RandomForest R²:   0.4987
Δ RMSE (tree vs baseline): +194.61 сек (лучше baseline)

=== MAE по срезам длительности (сек / мин) ===
+---------------+-------+------------+----------------+--------------+
|duration_bucket|matches|tree_MAE_min|baseline_MAE_min|avg_actual_min|
+---------------+-------+------------+----------------+--------------+
|Long (>45m)    |520    |10.28       |15.84           |52.21         |
|Medium (30-45m)|1226   |4.97        |3.66            |37.19         |
|Short (<30m)   |770    |4.79        |12.72           |23.65         |
+---------------+-------+------------+----------------+--------------+


=== Топ фичей по важности ===
          feature  importance
 gold_r

Rows after prep: 13048
Train: 10532, Test: 2516
Baseline avg duration: 2181.83 сек (36.36 мин.)

=== Сравнение кандидатов на тесте ===


rf_default    RMSE=  488.8  MAE=  374.3  R²=0.4943  (6.1s)


26/11/14 07:25:25 WARN DAGScheduler: Broadcasting large task binary with size 1079.0 KiB
26/11/14 07:25:26 WARN DAGScheduler: Broadcasting large task binary with size 2028.4 KiB
26/11/14 07:25:29 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/11/14 07:25:31 WARN DAGScheduler: Broadcasting large task binary with size 1040.0 KiB
26/11/14 07:25:32 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/11/14 07:25:35 WARN DAGScheduler: Broadcasting large task binary with size 1639.4 KiB
26/11/14 07:25:36 WARN DAGScheduler: Broadcasting large task binary with size 9.8 MiB
26/11/14 07:25:40 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/11/14 07:25:43 WARN DAGScheduler: Broadcasting large task binary with size 14.4 MiB
26/11/14 07:25:48 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/11/14 07:25:51 WARN DAGScheduler: Broadcasting large task binary with size 19.9 MiB
26/11/14 07:25:57 WARN DAGScheduler: Bro

rf_tuned      RMSE=  486.6  MAE=  371.9  R²=0.4988  (53.4s)


26/11/14 07:27:11 WARN DAGScheduler: Broadcasting large task binary with size 1003.1 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1006.3 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1006.7 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1007.7 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1008.5 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1010.9 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1015.6 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1018.4 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1018.9 KiB
26/11/14 07:27:12 WARN DAGScheduler: Broadcasting large task binary with size 1019.9 KiB
26/11/14 07:27:13 WARN DAGScheduler: Broadcasting large task binary with size 1020.7 KiB
26/11/14 07:27:13 WAR

gbt_tuned     RMSE=  506.5  MAE=  385.3  R²=0.4572  (75.1s)
baseline      RMSE=  687.5  MAE=  549.6  R²=-0.0003

🏆 Победитель: rf_tuned (RMSE=486.6)


✅ mart записан
[publish] attempt 1/5: mart.dota_duration_15min_match_duration_predictions
